[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoseGG8/AutoEncoderPi/blob/main/NuevoPipeline/ETL.ipynb)

# 🔬 Nuevo Pipeline ETL: Extracción Pura de Señales de Vibración (Paderborn 64 kHz)
### Descarga y Ventaneado con FFT de Motores Sanos (Buenos) y con Falla (Malos) — Datos Físicos Crudos

---

## 🎯 Objetivo de este Nuevo Pipeline
El objetivo de este notebook es construir un **dataset base 100% puro**, extrayendo la señal del canal de **vibraciones (`vibration_1`)** tanto de motores sanos como de motores con fallas reales, con una proporción controlada (**más motores sanos que defectuosos**).

A diferencia del pipeline anterior:
* **Cero transformaciones destructivas:** NO se aplica `log1p`, NO se aplica `Clipping` (winsorización), NO se aplica `RobustScaler` ni normalización cuántica $L_2$.
* **Canal físico:** Se utiliza exclusivamente el canal de acelerometría / vibraciones (**`vibration_1`**), donde los impactos de rodamiento generan armónicos mecánicos de alta frecuencia mucho más visibles que en corriente eléctrica.
* **Separación en Google Drive:** Los artefactos resultantes se almacenan en una carpeta independiente:
  `MyDrive/Paderborn_QML/NuevoPipeline/`
  ubicada de forma paralela a `PipeLine_Log1p_clipping_RobustScaling` para garantizar aislamiento absoluto entre experimentos.

---

### 📊 Estructura de Características Extraídas por Ventana (2048 muestras a 64 kHz):
1. **Condiciones de Operación (3):** `RPM`, `Torque_Nm`, `Fuerza_N`.
2. **Estadísticas Temporales de Vibración (4):** `RMS`, `Kurtosis`, `Skewness`, `Crest_Factor`.
3. **Espectro de Frecuencia FFT (64):** `fft_0` hasta `fft_63` (magnitud pura de la transformada de Fourier $|\mathcal{F}\{x - \bar{x}\}|$).
4. **Etiqueta Objetivo (1):** `Label` (`0` = Sano/Bueno, `1` = Falla/Malo).

In [ ]:
# ==============================================================================
# CELDA 1: ENTORNO, RUTAS Y CONFIGURACIÓN DEL NUEVO PIPELINE
# ==============================================================================
import os
import sys
import shutil
import urllib.request
import subprocess
import numpy as np
import pandas as pd
import scipy.io
from scipy.stats import kurtosis, skew

# Instalación de dependencias para descompresión RAR si es necesario
try:
    import rarfile
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rarfile"])
    import rarfile

if sys.platform.startswith("linux") and not shutil.which("unrar"):
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=False)

# Detección automática Colab vs entorno Local
EN_COLAB = os.path.exists("/content") and ("google.colab" in sys.modules or os.path.exists("/content/drive"))

# Configuración de carpetas en Google Drive
CARPETA_RAIZ_DRIVE = "Paderborn_QML"
NOMBRE_NUEVO_PIPELINE = "NuevoPipeline"

WORK_DIR = "/content" if EN_COLAB else os.getcwd()
TEMP_DIR = os.path.join(WORK_DIR, "temp_paderborn_vibraciones")
os.makedirs(TEMP_DIR, exist_ok=True)

DRIVE_DIR = None
if EN_COLAB:
    from google.colab import drive
    drive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"
    if not os.path.exists(drive_root):
        drive.mount("/content/drive")
        drive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"
    
    # Crear carpeta exclusiva del nuevo pipeline al lado de la anterior
    DRIVE_DIR = os.path.join(drive_root, CARPETA_RAIZ_DRIVE, NOMBRE_NUEVO_PIPELINE)
    os.makedirs(DRIVE_DIR, exist_ok=True)
    
    # También respaldamos en la raíz si existe acceso directo
    DRIVE_ALT_DIR = os.path.join(drive_root, NOMBRE_NUEVO_PIPELINE)
    os.makedirs(DRIVE_ALT_DIR, exist_ok=True)

# Parámetros físicos del ventaneado
WINDOW_SIZE = 2048              # Tamaño de ventana (~32 ms a 64 kHz)
N_FFT_BINS = 64                 # Primeros 64 coeficientes de magnitud FFT
CANAL_SENAL = "vibration_1"     # Canal de acelerometría mecánica (vibraciones)

# Nombres de archivos de salida limpios
OUT_CSV_PURA = os.path.join(WORK_DIR, "paderborn_vibracion_raw.csv")
OUT_CSV_META = os.path.join(WORK_DIR, "paderborn_vibracion_metadatos.csv")

print(f"✅ Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
if DRIVE_DIR:
    print(f"📁 Carpeta de salida en Google Drive: {DRIVE_DIR}")
print(f"🎯 Canal seleccionado: {CANAL_SENAL} | Ventana: {WINDOW_SIZE} muestras | FFT Bins: {N_FFT_BINS}")

In [ ]:
# ==============================================================================
# CELDA 2: CATÁLOGO DE ZENODO Y SELECCIÓN (MÁS BUENOS QUE MALOS)
# ==============================================================================
ZENODO_CATALOG = {
    # --- MOTORES SANOS / BUENOS (Normales, Label = 0) ---
    "K001.rar": "https://zenodo.org/records/15845309/files/K001.rar?download=1",
    "K002.rar": "https://zenodo.org/records/15845309/files/K002.rar?download=1",
    "K003.rar": "https://zenodo.org/records/15845309/files/K003.rar?download=1",
    "K004.rar": "https://zenodo.org/records/15845309/files/K004.rar?download=1",
    "K005.rar": "https://zenodo.org/records/15845309/files/K005.rar?download=1",
    "K006.rar": "https://zenodo.org/records/15845309/files/K006.rar?download=1",

    # --- MOTORES CON FALLAS REALES / MALOS (Fatiga por vida acelerada, Label = 1) ---
    "KA04.rar": "https://zenodo.org/records/15845309/files/KA04.rar?download=1",
    "KA15.rar": "https://zenodo.org/records/15845309/files/KA15.rar?download=1",
    "KI04.rar": "https://zenodo.org/records/15845309/files/KI04.rar?download=1",
    "KI14.rar": "https://zenodo.org/records/15845309/files/KI14.rar?download=1",
    "KB23.rar": "https://zenodo.org/records/15845309/files/KB23.rar?download=1",
    "KB27.rar": "https://zenodo.org/records/15845309/files/KB27.rar?download=1",

    # --- MOTORES CON FALLAS ARTIFICIALES (EDM, Label = 1) ---
    "KA01.rar": "https://zenodo.org/records/15845309/files/KA01.rar?download=1",
    "KI01.rar": "https://zenodo.org/records/15845309/files/KI01.rar?download=1",
}

# SELECCIÓN ESTRATÉGICA: Más rodamientos buenos (4) que malos (3)
# Produce aproximadamente ~10,000 ventanas sanas vs ~7,500 ventanas con falla real.
ARCHIVOS_A_DESCARGAR = [
    # 🟢 BUENOS (Sanos - Label = 0)
    "K001.rar",
    "K002.rar",
    "K003.rar",
    "K004.rar",
    # (Opcional: puedes descomentar K005 y K006 para obtener aún más sanos)
    # "K005.rar",
    # "K006.rar",

    # 🔴 MALOS (Fallas Reales por Fatiga - Label = 1)
    "KA04.rar",  # Pista exterior
    "KI04.rar",  # Pista interior
    "KB23.rar",  # Falla combinada
]

n_buenos = sum(1 for f in ARCHIVOS_A_DESCARGAR if f.startswith("K0"))
n_malos = len(ARCHIVOS_A_DESCARGAR) - n_buenos
print(f"📋 Archivos seleccionados para procesar ({len(ARCHIVOS_A_DESCARGAR)} rodamientos en total):")
print(f"   • Buenos (Sanos): {n_buenos} rodamientos -> {[f for f in ARCHIVOS_A_DESCARGAR if f.startswith('K0')]}")
print(f"   • Malos (Fallas): {n_malos} rodamientos -> {[f for f in ARCHIVOS_A_DESCARGAR if not f.startswith('K0')]}")
print(f"   • Proporción: {n_buenos} buenos vs {n_malos} malos (balance mayoritario en sanos ✅)")

In [ ]:
# ==============================================================================
# CELDA 3: DESCARGA STREAMING Y DESCOMPRESIÓN AUTOMÁTICA
# ==============================================================================
def extraer_rar(rar_path, target_folder):
    """Descomprime un archivo .rar usando unrar nativo, tar o rarfile."""
    if shutil.which("unrar"):
        res = subprocess.run(["unrar", "x", "-o+", rar_path, target_folder],
                             capture_output=True, text=True)
        if res.returncode == 0:
            return "unrar"

    if shutil.which("tar"):
        res = subprocess.run(["tar", "-xf", rar_path, "-C", target_folder],
                             capture_output=True, text=True)
        if res.returncode == 0:
            return "tar"

    with rarfile.RarFile(rar_path) as rf:
        rf.extractall(path=target_folder)
    return "rarfile"


def download_and_extract_selected(file_list, catalog, target_folder):
    """Descarga con barra de progreso y descomprime los archivos RAR seleccionados."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    print(f"\n[1/3] Descargando {len(file_list)} archivos RAR en {target_folder}...")

    for idx, filename in enumerate(file_list, 1):
        if filename not in catalog:
            print(f"  [!] '{filename}' no está en el catálogo. Omitiendo.")
            continue

        rar_path = os.path.join(target_folder, filename)
        if not os.path.exists(rar_path):
            print(f"\n  [{idx}/{len(file_list)}] Descargando {filename} desde Zenodo...")
            req = urllib.request.Request(catalog[filename], headers=headers)
            try:
                with urllib.request.urlopen(req) as resp, open(rar_path, "wb") as out_f:
                    total = int(resp.headers.get("Content-Length", 0))
                    downloaded = 0
                    while True:
                        buf = resp.read(1024 * 1024)
                        if not buf:
                            break
                        downloaded += len(buf)
                        out_f.write(buf)
                        mb_d = downloaded / (1024 * 1024)
                        if total > 0:
                            pct = (downloaded / total) * 100
                            sys.stdout.write(f"\r      [⬇] {mb_d:.1f} / {total/(1024*1024):.1f} MB ({pct:.1f}%)")
                        else:
                            sys.stdout.write(f"\r      [⬇] {mb_d:.1f} MB")
                        sys.stdout.flush()
                print(f"\n      [✓] Descarga completa: {filename}")
            except Exception as e:
                print(f"\n      [X] Error descargando {filename}: {e}")
                if os.path.exists(rar_path):
                    os.remove(rar_path)
                continue
        else:
            print(f"  [{idx}/{len(file_list)}] [✓] Ya descargado: {filename}")

        # Descomprimir
        print(f"      [📂] Descomprimiendo {filename}...")
        try:
            metodo = extraer_rar(rar_path, target_folder)
            print(f"      [✓] Extraído exitosamente (método: {metodo})")
        except Exception as e:
            print(f"      [X] Error al descomprimir {filename}: {e}")

In [ ]:
# ==============================================================================
# CELDA 4: EXTRACCIÓN DIRECTA DE VIBRACIÓN (VENTANAS + FFT PURA)
# ==============================================================================
OP_COLS = ["RPM", "Torque_Nm", "Fuerza_N"]
STAT_COLS = ["RMS", "Kurtosis", "Skewness", "Crest_Factor"]
FFT_COLS = [f"fft_{i}" for i in range(N_FFT_BINS)]
FEATURE_COLS = OP_COLS + STAT_COLS + FFT_COLS
SCHEMA_COLS = FEATURE_COLS + ["Label"]


def parse_metadata_from_filename(file_path):
    """Extrae condiciones de operación y etiqueta física del nombre del archivo .mat."""
    name = os.path.basename(file_path).upper()
    rpm = 900 if "N09" in name else 1500
    torque = 0.1 if "M01" in name else 0.7
    force = 400 if "F04" in name else 1000

    if "K0" in name:
        fault_type, label = "Normal", 0
    elif "KA" in name:
        fault_type, label = "Pista_Exterior", 1
    elif "KI" in name:
        fault_type, label = "Pista_Interior", 1
    elif "KB" in name:
        fault_type, label = "Combinado", 1
    else:
        fault_type, label = "Falla", 1

    return rpm, torque, force, fault_type, label


def extraer_vector_1d(data_obj):
    """Aplana recursivamente estructuras anidadas de MATLAB a un array numpy 1D."""
    arr = data_obj
    while isinstance(arr, np.ndarray) and arr.dtype == object and arr.size > 0:
        arr = arr.flat[0]
    return np.ravel(arr).astype(np.float64)


def get_vibration_channel(mat_path, min_len=200000):
    """Extrae específicamente la señal de vibración / acelerómetro (64 kHz, ~256k muestras)."""
    mat = scipy.io.loadmat(mat_path)
    key = next(k for k in mat.keys() if not k.startswith("__"))
    y_channels = mat[key][0, 0]["Y"]
    n_channels = y_channels.shape[1] if y_channels.ndim > 1 else y_channels.shape[0]

    for i in range(n_channels):
        ch = y_channels[0, i] if y_channels.ndim > 1 else y_channels[i]
        ch_name = ""
        if "Name" in ch.dtype.names:
            name_val = ch["Name"]
            while isinstance(name_val, np.ndarray) and name_val.size > 0:
                name_val = name_val.flat[0]
            ch_name = str(name_val).lower()

        signal = extraer_vector_1d(ch["Data"])
        if len(signal) < min_len:
            continue

        # Identificar el canal de vibración mecánica
        if any(w in ch_name for w in ["vib", "acc", "vibration", "beschleunigung"]):
            return signal

    # Fallback: si no tiene nombre explícito, el primer canal de 64 kHz es vibración
    for i in range(n_channels):
        ch = y_channels[0, i] if y_channels.ndim > 1 else y_channels[i]
        signal = extraer_vector_1d(ch["Data"])
        if len(signal) >= min_len:
            return signal

    return None


def extraer_features_ventana_pura(seg, rpm, torque, force, label):
    """
    Calcula exclusivamente las estadísticas básicas y la FFT cruda sin ninguna modificación:
      • RMS: Valor cuadrático medio físico
      • Kurtosis: Curtosis de Fisher
      • Skewness: Asimetría
      • Crest_Factor: Pico / RMS
      • fft_0..fft_63: Magnitud absoluta pura de la FFT |rfft(seg - mean(seg))|[:64]
      • Label: 0 (Bueno / Sano) o 1 (Malo / Falla)
      (SIN log1p, SIN clipping, SIN RobustScaler, SIN L2)
    """
    rms = float(np.sqrt(np.mean(seg ** 2)))
    std = float(np.std(seg))
    if std > 1e-12:
        kurt = float(kurtosis(seg, fisher=True, bias=True))
        skw = float(skew(seg, bias=True))
    else:
        kurt, skw = 0.0, 0.0
    crest = float(np.max(np.abs(seg)) / (rms + 1e-12))

    # FFT directa (eliminando solo el offset DC de la ventana para ver armónicos)
    seg_centered = seg - np.mean(seg)
    fft_mag = np.abs(np.fft.rfft(seg_centered))[:N_FFT_BINS]

    rec = {
        "RPM": rpm,
        "Torque_Nm": torque,
        "Fuerza_N": force,
        "RMS": rms,
        "Kurtosis": kurt,
        "Skewness": skw,
        "Crest_Factor": crest,
    }
    for idx, val in enumerate(fft_mag):
        rec[f"fft_{idx}"] = float(val)
    rec["Label"] = int(label)
    return rec


def process_all_vibration_mats(input_dir, window_size=WINDOW_SIZE, archivos_obj=ARCHIVOS_A_DESCARGAR):
    """Recorre los archivos .mat extraídos y genera las ventanas puras."""
    codigos_obj = {os.path.splitext(f)[0].upper() for f in archivos_obj} if archivos_obj else None
    mat_files = []
    for root, _, files in os.walk(input_dir):
        for f in sorted(files):
            if f.lower().endswith(".mat") and not f.startswith("._"):
                if codigos_obj and not any(c in f.upper() for c in codigos_obj):
                    continue
                mat_files.append(os.path.join(root, f))

    print(f"\n[2/3] Procesando {len(mat_files)} archivos .mat (Canal='{CANAL_SENAL}')...")
    records, metas = [], []
    omitidos = 0

    for mat_path in mat_files:
        filename = os.path.basename(mat_path)
        rpm, torque, force, fault_type, label = parse_metadata_from_filename(mat_path)

        try:
            sig = get_vibration_channel(mat_path)
            if sig is None:
                omitidos += 1
                continue

            num_windows = len(sig) // window_size
            for w_idx in range(num_windows):
                seg = sig[w_idx * window_size : (w_idx + 1) * window_size]
                records.append(extraer_features_ventana_pura(seg, rpm, torque, force, label))
                metas.append({
                    "Archivo": filename,
                    "Ventana": w_idx,
                    "Tipo_Falla": fault_type,
                    "RPM": rpm,
                    "Torque_Nm": torque,
                    "Fuerza_N": force,
                    "Label": int(label),
                })
        except Exception as e:
            print(f"  [X] Error procesando {filename}: {e}")

    print(f"  [✓] Ventanas extraídas exitosamente: {len(records)}")
    if omitidos > 0:
        print(f"  [!] Archivos omitidos por no contener señal válida: {omitidos}")

    df_raw = pd.DataFrame(records, columns=SCHEMA_COLS)
    df_meta = pd.DataFrame(metas)
    return df_raw, df_meta

In [ ]:
# ==============================================================================
# CELDA 5: EJECUCIÓN, VALIDACIÓN Y EXPORTACIÓN A GOOGLE DRIVE
# ==============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print(" 🚀 INICIANDO ETL: EXTRACCIÓN CRUDA DE VIBRACIONES (PADERBORN) ")
    print("=" * 70)

    # 1. Descarga y descompresión de rodamientos sanos y defectuosos
    download_and_extract_selected(ARCHIVOS_A_DESCARGAR, ZENODO_CATALOG, TEMP_DIR)

    # 2. Extracción de ventanas del canal de acelerometría mecánica
    df_raw, df_meta = process_all_vibration_mats(TEMP_DIR, window_size=WINDOW_SIZE)

    if len(df_raw) == 0:
        raise RuntimeError("No se extrajeron registros. Revisa ARCHIVOS_A_DESCARGAR y la carpeta temporal.")

    # 3. Comprobación de integridad (Cero modificación)
    n_nulos = df_raw.isnull().sum().sum()
    n_infs = np.isinf(df_raw[FEATURE_COLS].values).sum()
    print(f"\n[3/3] Verificación de Calidad Física:")
    print(f"   • Dimensiones del dataset: {df_raw.shape[0]} ventanas × {df_raw.shape[1]} columnas")
    print(f"   • Valores nulos detectados: {n_nulos}")
    print(f"   • Valores infinitos detectados: {n_infs}")
    if n_nulos > 0 or n_infs > 0:
        df_raw = df_raw.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
        df_meta = df_meta.loc[df_raw.index].reset_index(drop=True)

    # 4. Exportación de artefactos a disco local
    df_raw.to_csv(OUT_CSV_PURA, index=False)
    df_meta.to_csv(OUT_CSV_META, index=False)
    print(f"\n💾 Archivos guardados localmente:")
    print(f"   • Dataset crudo:     {OUT_CSV_PURA}")
    print(f"   • Metadatos detalle: {OUT_CSV_META}")

    # 5. Copia directa a la nueva carpeta principal en Google Drive
    if DRIVE_DIR is not None:
        for r_local in (OUT_CSV_PURA, OUT_CSV_META):
            f_name = os.path.basename(r_local)
            shutil.copy2(r_local, os.path.join(DRIVE_DIR, f_name))
            if 'DRIVE_ALT_DIR' in locals() and DRIVE_ALT_DIR:
                shutil.copy2(r_local, os.path.join(DRIVE_ALT_DIR, f_name))
        print(f"\n📁 [✓] Copiados exitosamente a Google Drive:")
        print(f"   • Carpeta principal: {DRIVE_DIR}")

    # 6. Resumen del balance de datos (Confirmando más buenos que malos)
    print("\n" + "=" * 70)
    print(" RESUMEN DEL DATASET GENERADO (BALANCE DE CLASES)")
    print("=" * 70)
    conteo = df_raw["Label"].value_counts()
    n_sanos = conteo.get(0, 0)
    n_fallas = conteo.get(1, 0)
    pct_sanos = (n_sanos / len(df_raw)) * 100
    pct_fallas = (n_fallas / len(df_raw)) * 100

    print(f"  • Motores Buenos (Sanos, Label=0): {n_sanos:,} ventanas ({pct_sanos:.1f}%)")
    print(f"  • Motores Malos (Fallas, Label=1): {n_fallas:,} ventanas ({pct_fallas:.1f}%)")
    print(f"  • Total Ventanas:                  {len(df_raw):,}")

    print("\nDesglose por Tipo de Rodamiento:")
    print(df_meta["Tipo_Falla"].value_counts().to_string())

    print("\nDesglose por Condiciones de Operación (RPM / Torque):")
    print(pd.crosstab(df_raw["RPM"], df_raw["Torque_Nm"], margins=True))

    print("\nEstadísticas Físicas Crudas de Vibración (Muestra de columnas):")
    cols_muestra = ["RMS", "Kurtosis", "Skewness", "Crest_Factor", "fft_0", "fft_1", "fft_2", "fft_3"]
    print(df_raw.groupby("Label")[cols_muestra].median().round(4).to_string())